# 2-1. Advanced RAG: Chunking 개선

## 학습 목표

- 청크 크기에 따라 검색 결과가 어떻게 달라지는지 비교한다.
- PDF 매뉴얼에 적합한 청크 크기를 실험한다.

## 공통 전제

- 실습 데이터: `../data/공직자_민원응대_핵심_매뉴얼.pdf`
- 기본 모델: `gpt-4o-mini`
- 기본 임베딩 모델: `text-embedding-3-small`
- 벡터DB: Qdrant 로컬 인메모리 모드

> 이 노트북은 `src` 파일을 import하지 않고, 노트북 안의 코드만으로 실행되도록 구성한다.

In [1]:
from pathlib import Path
import re
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

load_dotenv(override=True, dotenv_path="../../.env")

MD_PATH = Path("../data/공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md")

if not MD_PATH.exists():
    raise FileNotFoundError(f"Markdown 파일을 찾을 수 없습니다: {MD_PATH}")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [2]:
def load_markdown_page_chunks(md_path: Path) -> list[Document]:
    """
    전처리된 RAG page chunk Markdown 파일을 Document 리스트로 읽는다.

    대상 파일 예:
    ../data/공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md

    기준:
    - ## Page n | section | topic 단위로 분리
    - HTML comment metadata는 제거
    - page, section, topic 메타데이터를 유지
    """
    text = md_path.read_text(encoding="utf-8")

    # 첫 번째 문서 제목 제거
    text = re.sub(r"^# .+?\n", "", text, count=1)

    # ## Page 단위로 분리
    parts = re.split(r"(?=^## Page\s+\d+\s*\|)", text, flags=re.MULTILINE)

    docs = []

    for part in parts:
        part = part.strip()

        if not part:
            continue

        header_match = re.match(
            r"^## Page\s+(\d+)\s*\|\s*([^|]+)\s*\|\s*(.+)$",
            part.splitlines()[0].strip()
        )

        if not header_match:
            continue

        page = int(header_match.group(1))
        section = header_match.group(2).strip()
        topic = header_match.group(3).strip()

        lines = part.splitlines()[1:]

        # HTML comment metadata 제거
        content_lines = [
            line for line in lines
            if not line.strip().startswith("<!--")
        ]

        content = "\n".join(content_lines).strip()

        if not content:
            continue

        docs.append(
            Document(
                page_content=content,
                metadata={
                    "source": md_path.name,
                    "page": page,
                    "section": section,
                    "topic": topic,
                    "chunk_type": "page",
                }
            )
        )

    return docs

In [6]:
def make_chunks(docs: list[Document], chunk_size: int, chunk_overlap: int) -> list[Document]:
    """페이지 단위 Markdown 문서를 검색용 청크로 분할한다."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n## ", "\n# ", "\n- ", "\n", ". ", " "]
    )

    chunks = splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
        chunk.metadata["chunk_size"] = chunk_size

    return chunks

## 1. 서로 다른 청크 크기 만들기

In [7]:
pages = load_markdown_page_chunks(MD_PATH)

small_chunks = make_chunks(pages, chunk_size=400, chunk_overlap=80)
medium_chunks = make_chunks(pages, chunk_size=700, chunk_overlap=120)
large_chunks = make_chunks(pages, chunk_size=1000, chunk_overlap=150)

print("page docs:", len(pages))
print("small:", len(small_chunks))
print("medium:", len(medium_chunks))
print("large:", len(large_chunks))

page docs: 16
small: 59
medium: 34
large: 26


## 2. 청크 크기별 벡터스토어 생성

In [8]:
small_vs = QdrantVectorStore.from_documents(
    documents=small_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="small_chunks",
    force_recreate=True
)

medium_vs = QdrantVectorStore.from_documents(
    documents=medium_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="medium_chunks",
    force_recreate=True
)

large_vs = QdrantVectorStore.from_documents(
    documents=large_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="large_chunks",
    force_recreate=True
)

## 3. 검색 결과 비교

In [9]:
question = "전화 중 욕설을 하면 어떻게 대응해야 하나요?"

stores = {
    "small": small_vs,
    "medium": medium_vs,
    "large": large_vs
}

for name, store in stores.items():
    print("\n" + "=" * 80)
    print(f"[{name} chunk 검색 결과]")
    docs = store.similarity_search(question, k=3)

    for doc in docs:
        print(doc.metadata)
        print(doc.page_content[:350].replace("\n", " "))
        print()


[small chunk 검색 결과]
{'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md', 'page': 6, 'section': '일반적인 민원 응대요령', 'topic': '반복전화', 'chunk_type': 'page', 'chunk_id': 11, 'chunk_size': 400, '_id': 'b2e4a0ec97b047d68b8949b3ddad467e', '_collection_name': 'small_chunks'}
- 위법행위로부터 담당자를 보호하는 것을 우선하고 부서원과 상황을 공유하여 개인이 아닌 부서 차원에서 대응 2-1 정당한 사유 없는 반복 전화 ※ 동일민원을 3회 이상 제기 핵심대응 다른민원 처리를 위해 통화 지속 곤란 안내 후 종료 ‣ ~~통화 지속 곤란을 설명한 후 상담종료~~ 정당한 행정처분에 불복하여 반복전화하는 경우 선생님, 말씀하신 민원은 정해진 규정과 절차에 따라 정상적으로 처리된 사항입니다. (또는 현재 처리 중에 있습니다.) 동일한 답변 외에는 도움을 드릴 수 없으므로 다른 민원처리를 위해 통화를 종료하오니 양해하여 주시기 바랍니다. ※ 국민권익위원회 고충민원 등 신청 안내 충분한 설명이 이루어졌음에도 

{'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md', 'page': 7, 'section': '일반적인 민원 응대요령', 'topic': '장시간 통화', 'chunk_type': 'page', 'chunk_id': 13, 'chunk_size': 400, '_id': '30d5024a2bc240039fb0b5fa49a1d70e', '_collection_name': 'small_chunks'}
2-2 정당한 사유 없는 장시간 통화 ※ 권장시간 20분 설정의 경우 핵심대응 용건위주 질문유도 및 장시간 전화상담 곤란 안내 2-4 폭언(욕설, 협박, 성희롱 등) 핵심대응 전화 종료 및 증거확보를 통한 후속조치(법적조치 등)로 경각심 부여 ‣ ~~민원전화 

“작은 청크는 잘 찾지만 너무 잘게 쪼개져서 답변에 필요한 절차가 끊길 수 있고, 큰 청크는 문맥은 많지만 관련 없는 내용까지 섞인다. 이 실험에서는 중간 크기 청크가 검색 정확도와 답변 생성에 필요한 문맥을 가장 균형 있게 제공한다.”

# chunk 선택 판단

| 구분     |                 설정 | 청크 수 | 판단                           |
| ------ | -----------------: | ---: | ---------------------------- |
| small  |   400 / overlap 80 |  70개 | 검색 정확도는 좋지만 문맥이 짧게 잘릴 위험이 있다 |
| medium |  700 / overlap 120 |  43개 | 검색 정확도와 문맥 보존의 균형이 가장 좋다     |
| large  | 1000 / overlap 150 |  32개 | 문맥은 풍부하지만 불필요한 내용이 함께 섞인다    |


## 청크 수 판단 기준
| 평가 기준      | small | medium | large    |
| ---------- | ----- | ------ | -------- |
| 검색 정확도     | 높음    | 높음     | 중간       |
| 문맥 보존      | 낮음    | 높음     | 매우 높음    |
| 불필요한 정보 포함 | 낮음    | 적절함    | 높음       |
| 답변 생성 안정성  | 보통    | 가장 좋음  | 보통       |
| 수업 실습 적합성  | 좋음    | 가장 좋음  | 설명용으로 좋음 |


## 핵심 정리

- 작은 청크는 정확한 구간을 찾기 좋지만 문맥이 부족할 수 있다.
- 큰 청크는 문맥이 풍부하지만 불필요한 내용이 섞일 수 있다.
- 매뉴얼형 문서는 제목, 단계, 항목을 고려한 청크 분할이 중요하다.